# 예제 04. 감성 분류 모델 학습
빅데이터프로그래밍 · 12주차

## 목표
- 영화 리뷰 데이터를 불러온다
- `Embedding → LSTM → Linear` 모델을 만든다
- 긍정·부정을 분류하고 틀린 문장을 확인한다

10주차까지의 이미지 분류와 구조가 같습니다. 입력이 문장으로 바뀐 것뿐입니다.

**런타임 > 런타임 유형 변경 > T4 GPU** 를 먼저 선택하세요.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import re, random
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

torch.manual_seed(42)
random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. 데이터 준비
IMDB 영화 리뷰를 씁니다. 다운로드에 1분쯤 걸립니다.


In [ ]:
import urllib.request, tarfile, os

URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
if not os.path.exists("aclImdb"):
    print("내려받는 중...")
    urllib.request.urlretrieve(URL, "aclImdb.tar.gz")
    with tarfile.open("aclImdb.tar.gz") as f:
        f.extractall(".")
    print("완료")
else:
    print("이미 있습니다")


In [ ]:
def load_split(split, n_per_class=2500):
    data = []
    for label, name in [(1, "pos"), (0, "neg")]:
        d = f"aclImdb/{split}/{name}"
        files = sorted(os.listdir(d))[:n_per_class]
        for fn in files:
            with open(os.path.join(d, fn), encoding="utf-8") as f:
                data.append((f.read(), label))
    random.shuffle(data)
    return data


train_data = load_split("train")
test_data  = load_split("test", n_per_class=1250)

print(f"학습 {len(train_data)}개 · 검증 {len(test_data)}개")
print("\n예시:")
print(" 라벨", train_data[0][1], "/", train_data[0][0][:200], "...")


## 2. 토큰화와 사전


In [ ]:
PAD, UNK = 0, 1

def tokenize(s):
    s = s.lower().replace("<br />", " ")
    s = re.sub(r"[^a-z0-9'\s]", " ", s)
    return s.split()


train_tokens = [tokenize(t) for t, _ in train_data]
test_tokens  = [tokenize(t) for t, _ in test_data]

counter = Counter(w for t in train_tokens for w in t)
print("전체 단어:", f"{sum(counter.values()):,}개")
print("서로 다른 단어:", f"{len(counter):,}개")

MAX_VOCAB = 10000
vocab = {"<PAD>": PAD, "<UNK>": UNK}
for w, _ in counter.most_common(MAX_VOCAB - 2):
    vocab[w] = len(vocab)

print("사전 크기:", len(vocab))
print("\n자주 나온 단어:", [w for w, _ in counter.most_common(10)])


## 3. 문장 길이 확인 — MAX_LEN 정하기


In [ ]:
lens = [len(t) for t in train_tokens]
print(f"평균 {np.mean(lens):.0f} · 중앙값 {np.median(lens):.0f} · 90분위 {np.percentile(lens, 90):.0f} · 최대 {max(lens)}")

plt.figure(figsize=(8, 3))
plt.hist(lens, bins=60, range=(0, 800))
plt.axvline(200, color="crimson", linestyle="--", label="MAX_LEN 200")
plt.title("리뷰 길이 분포"); plt.legend(); plt.grid(alpha=.3)
plt.show()


## 4. Dataset


In [ ]:
MAX_LEN = 200

class ReviewDataset(Dataset):
    def __init__(self, token_lists, labels, vocab, max_len=MAX_LEN):
        seqs = []
        for t in token_lists:
            ids = [vocab.get(w, UNK) for w in t][:max_len]
            ids += [PAD] * (max_len - len(ids))
            seqs.append(ids)
        self.x = torch.tensor(seqs)
        self.y = torch.tensor(labels)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.x[i], self.y[i]


train_ds = ReviewDataset(train_tokens, [l for _, l in train_data], vocab)
test_ds  = ReviewDataset(test_tokens,  [l for _, l in test_data],  vocab)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False)

xb, yb = next(iter(train_loader))
print("입력:", tuple(xb.shape), "→ (batch, 길이)")
print("정답:", tuple(yb.shape))


## 5. 모델 — Embedding → LSTM → Linear


In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, hidden=64, n_classes=2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden, n_classes)

    def forward(self, x):
        e = self.emb(x)                # (batch, 길이, emb_dim)
        out, _ = self.lstm(e)          # (batch, 길이, hidden)
        last = out[:, -1, :]           # 마지막 시점
        return self.fc(self.dropout(last))


model = SentimentLSTM(len(vocab)).to(device)
print(model)
print("\n파라미터:", f"{sum(p.numel() for p in model.parameters()):,}개")
print("그중 임베딩:", f"{sum(p.numel() for p in model.emb.parameters()):,}개")
print("출력:", tuple(model(xb.to(device)).shape))


임베딩이 파라미터의 대부분입니다. 10,000 × 64 = 640,000개입니다.


## 6. 학습


In [ ]:
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

def evaluate(loader):
    model.eval()
    loss_sum = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += loss_fn(out, y).item() * y.numel()
            correct += (out.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return loss_sum / total, correct / total


history = []
for epoch in range(1, 9):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    history.append((*evaluate(train_loader), *evaluate(test_loader)))
    print(f"epoch {epoch}  학습 {history[-1][1]:.4f}  검증 {history[-1][3]:.4f}")


In [ ]:
xs = range(1, len(history)+1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(xs, [h[0] for h in history], label="학습")
ax[0].plot(xs, [h[2] for h in history], label="검증")
ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(xs, [h[1] for h in history], label="학습")
ax[1].plot(xs, [h[3] for h in history], label="검증")
ax[1].set_title("accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


## 7. 예측 결과와 틀린 문장


In [ ]:
model.eval()
all_pred, all_true, all_prob = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        all_pred.append(out.argmax(dim=1).cpu())
        all_prob.append(torch.softmax(out, dim=1).cpu())
        all_true.append(y)

pred = torch.cat(all_pred); true = torch.cat(all_true); prob = torch.cat(all_prob)
print(f"검증 정확도: {(pred == true).float().mean():.4f}")


In [ ]:
LABELS = ["부정", "긍정"]
wrong = (pred != true).nonzero().flatten()
print("틀린 개수:", len(wrong), "/", len(true))

print("\n틀린 리뷰 5개:")
for i in wrong[:5]:
    text = test_data[i][0][:160].replace("<br />", " ")
    print(f"\n  예측 {LABELS[pred[i]]} ({prob[i, pred[i]]:.2f}) / 정답 {LABELS[true[i]]}")
    print(f"  {text} ...")


## 8. 짧은 문장과 긴 문장의 차이


In [ ]:
test_lens = np.array([min(len(t), MAX_LEN) for t in test_tokens])
correct = (pred == true).numpy()

bins = [(0, 50), (50, 100), (100, 150), (150, 201)]
rows = []
for lo, hi in bins:
    mask = (test_lens >= lo) & (test_lens < hi)
    if mask.sum():
        rows.append({"길이 구간": f"{lo}~{hi}", "개수": int(mask.sum()),
                     "정확도": round(correct[mask].mean(), 4)})
df = pd.DataFrame(rows)
print(df.to_string(index=False))

plt.figure(figsize=(7, 3.4))
plt.bar(df["길이 구간"], df["정확도"])
plt.ylim(0, 1); plt.ylabel("정확도"); plt.title("문장 길이별 정확도")
plt.show()


짧은 리뷰는 판단할 근거가 적어 정확도가 낮은 경우가 많습니다. 뒤를 PAD로 채운 영향도 있습니다.


## 9. 직접 쓴 문장으로 예측해 보기


In [ ]:
def predict(text):
    ids = [vocab.get(w, UNK) for w in tokenize(text)][:MAX_LEN]
    n_unk = sum(1 for i in ids if i == UNK)
    ids += [PAD] * (MAX_LEN - len(ids))
    x = torch.tensor([ids]).to(device)
    model.eval()
    with torch.no_grad():
        p = torch.softmax(model(x), dim=1).cpu().squeeze()
    return LABELS[p.argmax()], p.max().item(), n_unk


tests = [
    "This movie was absolutely wonderful and I loved every minute of it",
    "Terrible film, boring plot and awful acting",
    "not bad at all, actually quite enjoyable",
    "I wanted to like it but it just did not work",
]

for t in tests:
    label, conf, n_unk = predict(t)
    print(f"{label} ({conf:.2f})  UNK {n_unk}개  ← {t}")


In [ ]:
torch.save({"model": model.state_dict(), "vocab": vocab}, "sentiment_lstm.pt")
print("저장 완료")


## 직접 해보기
1. `MAX_LEN` 을 400으로 늘리면 정확도가 오르나요?
2. 사전 크기를 3,000으로 줄이면 UNK가 얼마나 늘고 정확도는 어떻게 되나요?
3. LSTM을 양방향(`bidirectional=True`)으로 바꿔 보세요.


In [ ]:
# 여기에 작성하세요
